<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.5_Praktikum_Contour_Detection.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.5: Praktikum Contour Detection
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mengekstrak kontur batas topologis menggunakan `cv2.findContours` dengan bendera `RETR_EXTERNAL`.
2. Menghitung luas area (`contourArea`), keliling (`arcLength`), dan koordinat titik pusat massa (*centroid*) berbasis momen spasial.
3. Menguji deskriptor kebulatan (*circularity*) untuk membedakan tajuk kelapa sawit sehat dan tajuk terserang hama.
4. Memvisualisasikan anotasi kontur, bounding box, dan titik centroid pada data simulasi kebun.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')


## 1. Pembangkitan Masker Sensus Tajuk Pohon Sawit Sintetis
Mensimulasikan 4 tajuk pohon sawit (3 tajuk sehat berbentuk lingkaran simetris dan 1 tajuk rusak terserang hama) disertai derau rumput liar.

In [ ]:
H, W = 500, 700
orchard_mask = np.zeros((H, W), dtype=np.uint8)

# Pohon 1, 2, 3: Tajuk Sehat (Lingkaran Besar Simetris)
cv2.circle(orchard_mask, (180, 160), 65, 255, -1)
cv2.circle(orchard_mask, (420, 150), 60, 255, -1)
cv2.circle(orchard_mask, (250, 360), 70, 255, -1)

# Pohon 4: Tajuk Rusak / Terpapar Hama (Bentuk Tak Beraturan)
damaged_pts = np.array([[500, 310], [580, 320], [560, 400], [510, 420], [470, 360]], dtype=np.int32)
cv2.fillPoly(orchard_mask, [damaged_pts], 255)

# Derau semak belukar kecil (< 100 piksel)
cv2.circle(orchard_mask, (80, 400), 5, 255, -1)
cv2.circle(orchard_mask, (620, 200), 7, 255, -1)

print(f'Dimensi Citra Sensus Tajuk: {orchard_mask.shape}')


## 2. Pelacakan Kontur dan Ekstraksi Parameter Morfometri
Menerapkan `cv2.findContours` dan menghitung momen spasial, luas, keliling, serta kebulatan untuk setiap kontur tervalidasi.

In [ ]:
contours, hierarchy = cv2.findContours(orchard_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
MIN_AREA = 800

tree_data = []
canvas_vis = cv2.cvtColor(orchard_mask, cv2.COLOR_GRAY2BGR)

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < MIN_AREA:
        continue # Abaikan derau semak
        
    perim = cv2.arcLength(cnt, closed=True)
    circ = (4 * np.pi * area) / (perim ** 2) if perim > 0 else 0
    
    M = cv2.moments(cnt)
    cx = int(M['m10'] / M['m00']) if M['m00'] != 0 else 0
    cy = int(M['m01'] / M['m00']) if M['m00'] != 0 else 0
    
    is_healthy = circ >= 0.75
    label = 'Pohon Sehat' if is_healthy else 'Tajuk Rusak'
    color = (0, 255, 0) if is_healthy else (0, 0, 255)
    
    # Gambar visualisasi
    cv2.drawContours(canvas_vis, [cnt], -1, color, 2)
    cv2.circle(canvas_vis, (cx, cy), 5, (255, 0, 0), -1)
    
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(canvas_vis, (x, y), (x+w, y+h), (255, 200, 0), 1)
    
    tree_data.append({
        'id': len(tree_data) + 1,
        'centroid': (cx, cy),
        'area': area,
        'circularity': circ,
        'status': label
    })

print(f'Total Pohon Teridentifikasi: {len(tree_data)}')
for t in tree_data:
    print(f"ID {t['id']}: Centroid={t['centroid']}, Area={t['area']:.0f}, Kebulatan={t['circularity']:.3f} [{t['status']}]")


## 3. Visualisasi Peta Sensus Kanopi Kebun Kelapa Sawit
Menampilkan hasil segmentasi kontur, titik centroid, dan bounding box pohon sawit.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.imshow(orchard_mask, cmap='gray')
ax1.set_title('1. Masker Biner Sensus Tajuk Kebun', fontsize=11, fontweight='bold')
ax1.axis('off')

ax2.imshow(cv2.cvtColor(canvas_vis, cv2.COLOR_BGR2RGB))
ax2.set_title('2. Anotasi Kontur (Hijau=Sehat, Merah=Rusak)', fontsize=11, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.savefig('praktikum_sensus_kontur_11_5.png', dpi=150)
plt.show()
